# AlphaVedha — Full Model Training Pipeline

Train all 8 models on Nifty 50 data using GPU (Colab T4).

**Dependency order:**
1. Fetch OHLCV data via yfinance (no DB needed)
2. Compute 159 features + triple barrier labels per symbol
3. 3-way temporal split: Train / OOF / Validation (20-day embargo)
4. Train XGBoost → feature importance for LSTM/TFT
5. Train LSTM (2-layer, 128 hidden, 60-day sequences)
6. Train TFT (multi-horizon: 7d/15d/30d)
7. Train HMM Regime Detector (4-state)
8. Train Stacking Ensemble (RidgeClassifier on OOF predictions)
9. Train Meta-Labeling Model (binary gate, filters low-confidence)
10. Train Conformal Predictor (MAPIE, 90% coverage)
11. Zip artifacts → download

**Runtime:** ~30-60 min on Colab T4 GPU

## 0. Setup — Install AlphaVedha (Private Repo)

### Step 1: Create a GitHub Personal Access Token (one-time)

1. Open [GitHub Fine-Grained Token Settings](https://github.com/settings/personal-access-tokens/new)
2. Fill in:
   - **Token name:** `colab-training`
   - **Expiration:** 30 days (or custom)
   - **Resource owner:** `saurabhborkar22`
   - **Repository access:** Select **Only select repositories** → choose `alphavedha`
   - **Permissions → Repository permissions → Contents:** set to **Read-only**
3. Click **Generate token**
4. Copy the token (starts with `github_pat_...`) — save it somewhere safe, GitHub won't show it again

### Step 2: Run on Google Colab

1. Open [Google Colab](https://colab.research.google.com/)
2. **File → Upload notebook** → upload this `.ipynb` file
3. **Runtime → Change runtime type → T4 GPU** (free tier)
4. Run the cell below — it will ask you to paste your PAT
5. Run all remaining cells (Runtime → Run all)
6. When training finishes, the model zip auto-downloads to your browser

### Step 3: Load artifacts locally

```bash
cd ~/alphavedha
unzip ~/Downloads/alphavedha_models.zip -d models/artifacts/
alphavedha predict TCS      # test inference
alphavedha serve             # start API
```

In [ ]:
import os
from getpass import getpass

# Securely prompt for GitHub token (never stored in notebook)
token = getpass("Paste your GitHub PAT: ")
os.environ["GH_TOKEN"] = token

!git clone https://{token}@github.com/saurabhborkar22/alphavedha.git /content/alphavedha
%cd /content/alphavedha
!pip install -e . -q

# Clear token from memory
del token
os.environ.pop("GH_TOKEN", None)
print("Setup complete — token cleared from memory")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yfinance as yf

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — LSTM/TFT training will be slow")

## 1. Configuration

In [ ]:
# Training parameters
TIER = "large"  # "large" = Nifty 50
START_DATE = date(2020, 1, 1)
END_DATE = date.today()
OOF_RATIO = 0.15
VAL_RATIO = 0.15
EMBARGO_DAYS = 20
TOP_N_FEATURES = 30  # For LSTM/TFT feature selection

ARTIFACTS_DIR = Path("models/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Nifty 50 symbols (yfinance uses .NS suffix)
NIFTY_50 = [
    "HDFCBANK", "ICICIBANK", "SBIN", "KOTAKBANK", "AXISBANK", "INDUSINDBK",
    "TCS", "INFY", "WIPRO", "HCLTECH", "TECHM", "LTIM",
    "RELIANCE", "ONGC", "NTPC", "POWERGRID", "BPCL", "COALINDIA",
    "MARUTI", "TATAMOTORS", "M&M", "BAJAJ-AUTO", "HEROMOTOCO", "EICHERMOT",
    "SUNPHARMA", "DRREDDY", "CIPLA", "DIVISLAB", "APOLLOHOSP",
    "HINDUNILVR", "ITC", "NESTLEIND", "BRITANNIA", "TATACONSUM",
    "TATASTEEL", "JSWSTEEL", "HINDALCO", "ADANIENT",
    "BAJFINANCE", "BAJAJFINSV", "SBILIFE", "HDFCLIFE",
    "LT", "ADANIPORTS", "ULTRACEMCO", "GRASIM", "SHRIRAMFIN",
    "BHARTIARTL", "ASIANPAINT", "TITAN",
]

print(f"Symbols: {len(NIFTY_50)}")
print(f"Date range: {START_DATE} → {END_DATE}")

## 2. Fetch OHLCV Data

In [ ]:
def fetch_ohlcv(symbol: str, start: date, end: date) -> pd.DataFrame:
    """Fetch OHLCV from yfinance with .NS suffix for NSE stocks."""
    ticker = f"{symbol}.NS"
    df = yf.download(ticker, start=str(start), end=str(end), progress=False, auto_adjust=False)
    if df.empty:
        return df
    # Flatten multi-level columns from yfinance
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    df = df.rename(columns={
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Adj Close": "adj_close", "Volume": "volume",
    })
    df.index.name = "date"
    return df


print("Fetching OHLCV data for all symbols...")
ohlcv_data: dict[str, pd.DataFrame] = {}
failed: list[str] = []

for i, symbol in enumerate(NIFTY_50):
    try:
        df = fetch_ohlcv(symbol, START_DATE, END_DATE)
        if len(df) >= 252:
            ohlcv_data[symbol] = df
            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(NIFTY_50)}] {symbol}: {len(df)} rows")
        else:
            failed.append(f"{symbol} ({len(df)} rows)")
    except Exception as e:
        failed.append(f"{symbol} ({e})")

print(f"\nLoaded: {len(ohlcv_data)} symbols")
if failed:
    print(f"Skipped: {failed}")

## 3. Compute Features & Labels

In [ ]:
from alphavedha.config import get_config
from alphavedha.features.pipeline import compute_all_features
from alphavedha.labels.triple_barrier import compute_triple_barrier_labels

config = get_config()


def prepare_symbol(symbol: str, ohlcv_df: pd.DataFrame):
    """Compute features + labels for one symbol."""
    feature_result = compute_all_features(symbol=symbol, ohlcv_df=ohlcv_df)
    label_result = compute_triple_barrier_labels(
        ohlcv_df, config.labels.triple_barrier, symbol=symbol,
    )
    features_df = feature_result.df
    labels_df = label_result.df

    valid_mask = labels_df["label"].notna()
    features_df = features_df.loc[valid_mask]
    labels_df = labels_df.loc[valid_mask]

    if len(features_df) < 100:
        return None
    return features_df, labels_df["label"].astype(int), labels_df["return_pct"]


print("Computing features and labels...")
start_t = time.perf_counter()

all_prepared: list[tuple[pd.DataFrame, pd.Series, pd.Series]] = []
symbol_errors: dict[str, str] = {}

for i, (symbol, ohlcv_df) in enumerate(ohlcv_data.items()):
    try:
        result = prepare_symbol(symbol, ohlcv_df)
        if result is not None:
            all_prepared.append(result)
            if (i + 1) % 10 == 0:
                print(f"  [{i+1}/{len(ohlcv_data)}] {symbol}: {len(result[0])} rows, {result[0].shape[1]} features")
        else:
            symbol_errors[symbol] = "too few valid labels"
    except Exception as e:
        symbol_errors[symbol] = str(e)
        print(f"  ERROR {symbol}: {e}")

elapsed = time.perf_counter() - start_t
print(f"\nPrepared {len(all_prepared)} symbols in {elapsed:.1f}s")
if symbol_errors:
    print(f"Errors: {symbol_errors}")

## 4. Temporal Split (Train / OOF / Val)

In [ ]:
from alphavedha.training.pipeline import _temporal_split_3way, TierData

all_train, all_oof, all_val = [], [], []

for features_df, y_dir, y_ret in all_prepared:
    X_tr, y_tr, ret_tr, X_o, y_o, ret_o, X_v, y_v, ret_v = _temporal_split_3way(
        features_df, y_dir, y_ret,
        oof_ratio=OOF_RATIO, val_ratio=VAL_RATIO, embargo_days=EMBARGO_DAYS,
    )
    if len(X_tr) > 0:
        all_train.append((X_tr, y_tr, ret_tr))
    if len(X_o) > 0:
        all_oof.append((X_o, y_o, ret_o))
    if len(X_v) > 0:
        all_val.append((X_v, y_v, ret_v))


def concat_parts(parts):
    if not parts:
        return pd.DataFrame(), pd.Series(dtype=float), pd.Series(dtype=float)
    Xs, ys, rs = zip(*parts, strict=True)
    return pd.concat(Xs, ignore_index=True), pd.concat(ys, ignore_index=True), pd.concat(rs, ignore_index=True)


X_train, y_train, ret_train = concat_parts(all_train)
X_oof, y_oof, ret_oof = concat_parts(all_oof)
X_val, y_val, ret_val = concat_parts(all_val)

# Build TierData for helpers that need it
data = TierData(
    X_train=X_train, y_train=y_train, ret_train=ret_train,
    X_oof=X_oof, y_oof=y_oof, ret_oof=ret_oof,
    X_val=X_val, y_val=y_val, ret_val=ret_val,
    ohlcv_by_symbol=ohlcv_data,
    n_symbols=len(all_prepared),
    errors=symbol_errors,
)

print(f"Train: {len(X_train):,} rows, {X_train.shape[1]} features")
print(f"OOF:   {len(X_oof):,} rows")
print(f"Val:   {len(X_val):,} rows")
print(f"Labels: {dict(y_train.value_counts().sort_index())}")

## 5. Train XGBoost

In [ ]:
from alphavedha.models.xgboost_model import XGBoostModel

print("Training XGBoost...")
start_t = time.perf_counter()

xgb_model = XGBoostModel(config=config.models.xgboost)
xgb_result = xgb_model.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    return_train=ret_train, return_val=ret_val,
)

xgb_dir = ARTIFACTS_DIR / "xgboost" / "latest"
xgb_model.save(xgb_dir)

elapsed = time.perf_counter() - start_t
print(f"  Train accuracy: {xgb_result.train_metrics.get('accuracy', 0):.4f}")
print(f"  Val accuracy:   {xgb_result.val_metrics.get('accuracy', 0):.4f}")
print(f"  Val F1:         {xgb_result.val_metrics.get('f1_weighted', 0):.4f}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {xgb_dir}")

In [ ]:
# Feature importance → used for LSTM/TFT feature selection
from alphavedha.training.pipeline import _select_top_features

fi = xgb_model.get_feature_importance()
top_features = _select_top_features(fi, TOP_N_FEATURES, list(X_train.columns)) if fi is not None else list(X_train.columns)[:TOP_N_FEATURES]

print(f"Top {len(top_features)} features for LSTM/TFT:")
for i, f in enumerate(top_features[:10]):
    print(f"  {i+1}. {f} (importance: {fi[f]:.4f})")
print(f"  ... and {len(top_features) - 10} more")

## 6. Train LSTM

In [ ]:
from alphavedha.models.lstm_model import LSTMModel
from alphavedha.training.pipeline import _fill_nan_for_torch

print("Training LSTM (2-layer, 128 hidden, 60-day sequences)...")
start_t = time.perf_counter()

X_train_lstm = _fill_nan_for_torch(X_train[top_features])
X_val_lstm = _fill_nan_for_torch(X_val[top_features])

lstm_model = LSTMModel(config=config.models.lstm)
lstm_result = lstm_model.fit(
    X_train_lstm, y_train,
    X_val=X_val_lstm, y_val=y_val,
    return_train=ret_train, return_val=ret_val,
)

lstm_dir = ARTIFACTS_DIR / "lstm" / "latest"
lstm_model.save(lstm_dir)

elapsed = time.perf_counter() - start_t
print(f"  Train accuracy: {lstm_result.train_metrics.get('accuracy', 0):.4f}")
print(f"  Val accuracy:   {lstm_result.val_metrics.get('accuracy', 0):.4f}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {lstm_dir}")

## 7. Train TFT (Temporal Fusion Transformer)

In [ ]:
from alphavedha.models.temporal_attention import TemporalAttentionModel

print("Training TFT (multi-horizon: 7d/15d/30d)...")
start_t = time.perf_counter()

X_train_tft = _fill_nan_for_torch(X_train[top_features])
X_val_tft = _fill_nan_for_torch(X_val[top_features])

tft_model = TemporalAttentionModel(config=config.models.tft)
tft_result = tft_model.fit(
    X_train_tft, y_train,
    X_val=X_val_tft, y_val=y_val,
    return_train=ret_train, return_val=ret_val,
)

tft_dir = ARTIFACTS_DIR / "tft" / "latest"
tft_model.save(tft_dir)

elapsed = time.perf_counter() - start_t
print(f"  Train accuracy: {tft_result.train_metrics.get('accuracy', 0):.4f}")
print(f"  Val accuracy:   {tft_result.val_metrics.get('accuracy', 0):.4f}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {tft_dir}")

## 8. Train HMM Regime Detector

In [ ]:
from alphavedha.models.regime import RegimeDetector

print("Training HMM Regime Detector (4-state: bull/bear/sideways/high-vol)...")
start_t = time.perf_counter()

all_returns = []
for symbol, ohlcv_df in ohlcv_data.items():
    if "close" in ohlcv_df.columns and len(ohlcv_df) > 50:
        rets = np.log(ohlcv_df["close"] / ohlcv_df["close"].shift(1)).dropna()
        all_returns.append(rets)

combined = pd.concat(all_returns, axis=1)
portfolio_returns = combined.mean(axis=1).dropna()
realized_vol = portfolio_returns.rolling(20).std().dropna()
portfolio_returns = portfolio_returns.loc[realized_vol.index]

detector = RegimeDetector(config=config.models.regime)
regime_metrics = detector.fit(portfolio_returns, realized_vol)

regime_dir = ARTIFACTS_DIR / "regime" / "latest"
detector.save(regime_dir)

elapsed = time.perf_counter() - start_t
print(f"  Metrics: {regime_metrics}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {regime_dir}")

## 9. Train Stacking Ensemble

In [ ]:
from alphavedha.models.ensemble import StackingEnsemble
from alphavedha.training.pipeline import _get_base_predictions, _get_regime_probs

print("Training Stacking Ensemble (RidgeClassifier on OOF predictions)...")
start_t = time.perf_counter()

# Get OOF predictions from all 3 base models
base_preds_oof = _get_base_predictions(
    xgb_model, lstm_model, tft_model,
    X_oof, top_features,
)
regime_probs_oof = _get_regime_probs(data, len(X_oof))

ensemble = StackingEnsemble(config=config.models.ensemble)
ens_metrics = ensemble.fit(base_preds_oof, regime_probs_oof, y_oof)

ens_dir = ARTIFACTS_DIR / "ensemble" / "latest"
ensemble.save(ens_dir)

elapsed = time.perf_counter() - start_t
print(f"  Metrics: {ens_metrics}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {ens_dir}")

## 10. Train Meta-Labeling Model

In [ ]:
from alphavedha.models.meta_model import MetaLabelingModel

print("Training Meta-Labeling Model (binary gate, threshold=0.55)...")
start_t = time.perf_counter()

# Get ensemble predictions on OOF data
ensemble_result = ensemble.predict(base_preds_oof, regime_probs_oof)

# Binary labels: was the ensemble prediction correct?
y_correct = pd.Series(
    (ensemble_result.direction == y_oof.values).astype(int),
    index=y_oof.index,
)

X_oof_clean = _fill_nan_for_torch(X_oof)

meta_model = MetaLabelingModel()
meta_metrics = meta_model.fit(
    X_oof_clean,
    ensemble_result.direction,
    ensemble_result.confidence,
    y_correct,
)

meta_dir = ARTIFACTS_DIR / "meta_labeling" / "latest"
meta_model.save(meta_dir)

elapsed = time.perf_counter() - start_t
print(f"  Metrics: {meta_metrics}")
print(f"  Filter rate: {meta_metrics.get('filter_rate', 0):.1%} of signals filtered")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {meta_dir}")

## 11. Train Conformal Predictor

In [ ]:
from alphavedha.models.conformal import ConformalPredictor

print("Training Conformal Predictor (MAPIE, 90% coverage)...")
start_t = time.perf_counter()

predictor = ConformalPredictor()
conf_metrics = predictor.fit(X_oof_clean, ret_oof)

conf_dir = ARTIFACTS_DIR / "conformal" / "latest"
predictor.save(conf_dir)

elapsed = time.perf_counter() - start_t
print(f"  Metrics: {conf_metrics}")
print(f"  Time: {elapsed:.1f}s")
print(f"  Saved to: {conf_dir}")

## 12. Validation Summary

In [ ]:
# Run all models on validation set for final metrics
print("=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)

# XGBoost val
xgb_val_pred = xgb_model.predict(X_val)
xgb_val_acc = (xgb_val_pred.direction == y_val.values).mean()
print(f"\nXGBoost val accuracy:  {xgb_val_acc:.4f}")

# LSTM val
lstm_val_pred = lstm_model.predict(_fill_nan_for_torch(X_val[top_features]))
lstm_val_acc = (lstm_val_pred.direction == y_val.values).mean()
print(f"LSTM val accuracy:     {lstm_val_acc:.4f}")

# TFT val
tft_val_pred = tft_model.predict(_fill_nan_for_torch(X_val[top_features]))
tft_val_acc = (tft_val_pred.direction == y_val.values).mean()
print(f"TFT val accuracy:      {tft_val_acc:.4f}")

# Ensemble val
base_preds_val = _get_base_predictions(xgb_model, lstm_model, tft_model, X_val, top_features)
regime_probs_val = _get_regime_probs(data, len(X_val))
ens_val = ensemble.predict(base_preds_val, regime_probs_val)
ens_val_acc = (ens_val.direction == y_val.values).mean()
print(f"Ensemble val accuracy: {ens_val_acc:.4f}")
print(f"Model disagreement:    {ens_val.model_disagreement.mean():.4f}")

# Meta-labeling filter
meta_val = meta_model.predict(
    _fill_nan_for_torch(X_val), ens_val.direction, ens_val.confidence,
)
tradeable_mask = meta_val.is_tradeable
if tradeable_mask.sum() > 0:
    filtered_acc = (ens_val.direction[tradeable_mask] == y_val.values[tradeable_mask]).mean()
    print(f"\nMeta-label filter rate: {(~tradeable_mask).mean():.1%}")
    print(f"Post-filter accuracy:  {filtered_acc:.4f}")
else:
    print("\nMeta-label filtered ALL signals (threshold too strict)")

print(f"\nSymbols: {data.n_symbols} | Train: {len(X_train):,} | OOF: {len(X_oof):,} | Val: {len(X_val):,}")

## 13. Download Artifacts

In [ ]:
import shutil

# List all saved artifacts
print("Saved model artifacts:")
for model_dir in sorted(ARTIFACTS_DIR.glob("*/latest")):
    files = list(model_dir.iterdir())
    total_size = sum(f.stat().st_size for f in files if f.is_file())
    print(f"  {model_dir.parent.name}: {len(files)} files, {total_size / 1024:.1f} KB")

# Zip for download
zip_path = "/content/alphavedha_models"
shutil.make_archive(zip_path, "zip", ARTIFACTS_DIR)
print(f"\nZipped to: {zip_path}.zip")
print(f"Size: {Path(zip_path + '.zip').stat().st_size / (1024*1024):.1f} MB")

In [ ]:
# Download in Colab
try:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("Download started — check your browser downloads")
except ImportError:
    print(f"Not running in Colab. Artifacts at: {ARTIFACTS_DIR}")
    print(f"Zip at: {zip_path}.zip")

## Usage After Download

```bash
# On your local machine:
cd alphavedha
unzip ~/Downloads/alphavedha_models.zip -d models/artifacts/

# Test inference
alphavedha predict TCS

# Start API with real models
alphavedha serve
```